In [2]:
"""
freeze_design.py  --  Design freeze v1 for the dual-filter qubit readout classifier.

What it does (everything the RTL needs, in one run):
  1. builds the data for each MODE   ("single": real QICK single-qubit data;  "bus5": SYNTHETIC 5-qubit multiplexed bus;
                                      "spin_single" / "spin_bus5": synthetic spin-qubit RF reflectometry datasets)
  2. calibrates the dual filter (WMF + WRMF, EMA) on the CAL split
  3. quantizes with the FROZEN bit widths and trains the classifier on integer features (CLF split)
  4. measures float-vs-fixed loss on the TEST split (paired, with standard error)  -> warns if the loss is too big
  5. exports .mem files (Vivado $readmemh format), config.json, design_spec.md and golden test vectors

Integer algorithm (this is exactly what the hardware must compute):
  acc[k]  = sum_t ( I[t]*A_k[t] + Q[t]*B_k[t] )                      k = 0..3 per qubit  (MF_I, MF_Q, RMF_I, RMF_Q)
  tmp     = clip( acc + 2^n - mu , 0 , 2^(n+1) )
  u       = (tmp << F) >> (n+1)           v = u - 2^(F-1)            (v in [-2^(F-1), +2^(F-1)] = [-0.5, +0.5])
  product = (v_i * v_j) >> F              (arithmetic shift = floor)
  score   = sum_i  w_i * o_i ,   o = [2^F, v..., products...]         decision = score > thr

Run verify_mem_files.py afterwards: it re-derives every golden output from the .mem files alone.
"""
import os
import json
import numpy as np

# ====================================================================== FROZEN PARAMETERS
DATA_DIR = r"D:\MSU\Qubit_readout_FPGA\python"
OUT_DIR = "design_freeze"
MODES = ["single", "bus5", "spin_single", "spin_bus5"]
BUS_STRENGTH = 1.0                  # crosstalk strength for the synthetic bus (1.0 = original model)
SEED = 42
T_BINS = 500                        # default accumulation length (bins) for superconducting modes
EMA_ALPHA = 0.05
CLIP = 99.99                        # normalizer amplitude percentile
LAMBDA_REL = 1e-6                   # ridge strength (relative to N)

INPUT_BITS = 16                     # int16 container for ADC samples (data peak is ~14 bits)
DLO_BITS = 10                       # signed width of the weight-table words
F_BITS = 12                         # fractional bits of normalized features (1.0 = 2^F)
W_BITS = 12                         # signed width of classifier weights
W_FRAC = 6                          # fractional bits of classifier weights
ALLOWED_LOSS_PTS = 0.05             # max acceptable (float - fixed) fidelity loss, in points

POOL_EACH = 100_000                 # single mode: CAL and CLF shots (disjoint pools from the train file)
N_CAL_BUS, N_CLF_BUS, N_TEST_BUS = 30_000, 50_000, 100_000
N_GOLD_NPZ = 2000                   # golden shots saved in the .npz
N_GOLD_MEM = 256                    # golden shots written as text .mem for the RTL testbench (+4 stress shots)
FS = 500.0e6                        # ASSUMED bus sample rate for the synthetic bus (real QICK rate still unverified)
IF_BUS = np.array([64.729e6, 25.366e6, -24.790e6, -70.269e6, -127.282e6])
VARIANTS = ["lin", "poly"]
MEM_WORD = 48                       # width used for acc / score / mu / threshold words in .mem files


def find_file(fname):
    for d in [DATA_DIR, os.path.join(DATA_DIR, "python"), "."]:
        cand = os.path.join(d, fname)
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(f"Cannot find {fname} in {DATA_DIR} or current folder.")


# ====================================================================== load source pools
Xtr = np.load(find_file("0528_X_train_0_770.npy"), mmap_mode="r")
Xte_m = np.load(find_file("0528_X_test_0_770.npy"), mmap_mode="r")
ytr = np.load(find_file("0528_y_train_0_770.npy")).astype(int)
yte = np.load(find_file("0528_y_test_0_770.npy")).astype(int)

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(ytr))
cal_pool = np.sort(perm[:POOL_EACH])
clf_pool = np.sort(perm[POOL_EACH:2 * POOL_EACH])
assert len(np.intersect1d(cal_pool, clf_pool)) == 0


def load_pool(X, idx):
    out = np.empty((len(idx), 2 * T_BINS), np.int16)
    for i in range(0, len(idx), 20_000):
        out[i:i + 20_000] = np.rint(X[idx[i:i + 20_000], :2 * T_BINS]).astype(np.int16)
    return out


print("loading source-shot pools ...")
S_cal, y_cal_pool = load_pool(Xtr, cal_pool), ytr[cal_pool]
S_clf, y_clf_pool = load_pool(Xtr, clf_pool), ytr[clf_pool]
S_te, y_te_pool = load_pool(Xte_m, np.arange(len(yte))), yte
peak = int(max(np.abs(S_cal).max(), np.abs(S_clf).max(), np.abs(S_te).max()))
assert peak <= 2 ** (INPUT_BITS - 1) - 1, "data exceeds INPUT_BITS"
print(f"peak |sample| = {peak}  ({int(np.ceil(np.log2(peak + 1))) + 1} signed bits needed; container = {INPUT_BITS})")

t_sec = np.arange(T_BINS, dtype=np.float64) / FS
LIN = np.array([[1.00, 0.15, 0.04, 0.01, 0.00], [0.15, 1.00, 0.16, 0.04, 0.01], [0.04, 0.16, 1.00, 0.14, 0.04],
                [0.01, 0.04, 0.14, 1.00, 0.15], [0.00, 0.01, 0.04, 0.15, 1.00]])
PH = np.array([[0.00, 0.28, 0.08, 0.00, 0.00], [-0.26, 0.00, 0.30, 0.08, 0.00], [0.07, -0.28, 0.00, 0.26, 0.07],
               [0.00, 0.07, -0.25, 0.00, 0.28], [0.00, 0.00, 0.07, -0.27, 0.00]])


def synth(S, idx, Y, s, chunk=5000):
    n = len(idx)
    carriers = np.exp(1j * 2.0 * np.pi * IF_BUS[:, None] * t_sec[None, :])
    L = np.eye(5) + s * (LIN - np.eye(5))
    P = s * PH
    I_out = np.empty((n, T_BINS), np.int16)
    Q_out = np.empty((n, T_BINS), np.int16)
    for a in range(0, n, chunk):
        b = min(a + chunk, n)
        bb = np.empty((b - a, 5, T_BINS), np.complex128)
        for q in range(5):
            rows = S[idx[a:b, q]].astype(np.float64)
            bb[:, q, :] = rows[:, 0::2] + 1j * rows[:, 1::2]
        sgn = 2.0 * Y[a:b] - 1.0
        theta = sgn @ P.T
        prev = np.zeros_like(sgn)
        prev[:, 1:] = sgn[:, :-1]
        bb = bb * ((1.0 + 0.12 * s * prev)[:, :, None] * np.exp(1j * theta[:, :, None]))
        bb = np.einsum("pq,bqt->bpt", L, bb)
        bus = np.einsum("bkt,kt->bt", bb, carriers) / np.sqrt(5)
        I_out[a:b] = np.clip(np.rint(bus.real), -32768, 32767).astype(np.int16)
        Q_out[a:b] = np.clip(np.rint(bus.imag), -32768, 32767).astype(np.int16)
    return I_out, Q_out


def get_data(mode):
    if mode == "single":
        sp = lambda S: (np.ascontiguousarray(S[:, 0::2]), np.ascontiguousarray(S[:, 1::2]))
        d = dict(IF=np.array([0.0]), desc="REAL QICK single-qubit data (Zenodo 14427490), baseband (already downconverted)")
        d["cal"], d["clf"], d["te"] = sp(S_cal) + (y_cal_pool[:, None],), sp(S_clf) + (y_clf_pool[:, None],), sp(S_te) + (y_te_pool[:, None],)
        return d
    if mode == "bus5":
        r2 = np.random.default_rng(SEED + 1)
        d = dict(IF=IF_BUS, desc=f"SYNTHETIC 5-qubit multiplexed bus built from real single-qubit shots, crosstalk strength {BUS_STRENGTH}")
        for key, S, yp, n in (("cal", S_cal, y_cal_pool, N_CAL_BUS), ("clf", S_clf, y_clf_pool, N_CLF_BUS),
                              ("te", S_te, y_te_pool, N_TEST_BUS)):
            idx = r2.integers(0, len(yp), size=(n, 5))
            Y = yp[idx]
            I, Q = synth(S, idx, Y, BUS_STRENGTH)
            d[key] = (I, Q, Y)
        return d
    if mode in ("spin_single", "spin_bus5"):
        npz = np.load(find_file(os.path.join("data", "synth", f"{mode}.npz")))
        Xtr_s, ytr_s = npz["X_train"], npz["y_train"]
        Xte_s, yte_s = npz["X_test"], npz["y_test"]
        if ytr_s.ndim == 1:
            ytr_s = ytr_s[:, None]
            yte_s = yte_s[:, None]
        n_cal = int(0.4 * len(Xtr_s))
        d = dict(
            IF=npz["f_if"],
            desc=f"SYNTHETIC spin-qubit RF reflectometry dataset ({mode})"
        )
        d["cal"] = (np.ascontiguousarray(Xtr_s[:n_cal, :, 0]), np.ascontiguousarray(Xtr_s[:n_cal, :, 1]), ytr_s[:n_cal])
        d["clf"] = (np.ascontiguousarray(Xtr_s[n_cal:, :, 0]), np.ascontiguousarray(Xtr_s[n_cal:, :, 1]), ytr_s[n_cal:])
        d["te"] = (np.ascontiguousarray(Xte_s[:, :, 0]), np.ascontiguousarray(Xte_s[:, :, 1]), yte_s)
        return d
    raise ValueError(mode)


# ====================================================================== filters (float) and weight tables
def ema_1d(x, alpha):
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = alpha * x[t] + (1.0 - alpha) * out[t - 1]
    return out


def premix(q, w0I, w0Q, w1I, w1Q, W, IF, t_sec):
    c, sn = np.cos(2.0 * np.pi * IF[q] * t_sec), np.sin(2.0 * np.pi * IF[q] * t_sec)
    W[q, 0, 0], W[q, 0, 1] = w0I * c, w0I * sn          # feature 0 (MF_I): A multiplies I, B multiplies Q
    W[q, 1, 0], W[q, 1, 1] = -w0Q * sn, w0Q * c         # feature 1 (MF_Q)
    W[q, 2, 0], W[q, 2, 1] = w1I * c, w1I * sn          # feature 2 (RMF_I)
    W[q, 3, 0], W[q, 3, 1] = -w1Q * sn, w1Q * c         # feature 3 (RMF_Q)


def calibrate(I_raw, Q_raw, Y, IF, fs=FS):
    n_q, T = Y.shape[1], I_raw.shape[1]
    t_sec = np.arange(T, dtype=np.float64) / fs
    S_raw = I_raw.astype(np.float64) + 1j * Q_raw.astype(np.float64)
    W = np.zeros((n_q, 4, 2, T))
    relax_counts = []
    for q in range(n_q):
        Zq = S_raw * np.exp(-1j * 2.0 * np.pi * IF[q] * t_sec)[None, :]
        Iq, Qq, yq = Zq.real, Zq.imag, Y[:, q]
        m0, m1 = yq == 0, yq == 1
        mI0, mQ0, vI0, vQ0 = Iq[m0].mean(0), Qq[m0].mean(0), Iq[m0].var(0), Qq[m0].var(0)
        mI1, mQ1, vI1, vQ1 = Iq[m1].mean(0), Qq[m1].mean(0), Iq[m1].var(0), Qq[m1].var(0)
        mtv = np.c_[Iq.mean(1), Qq.mean(1)]
        C0, C1 = mtv[m0].mean(0), mtv[m1].mean(0)
        relax = m1 & (np.linalg.norm(mtv - C0, axis=1) < 0.5 * np.linalg.norm(C0 - C1))
        relax_counts.append(int(relax.sum()))
        wmI = ema_1d(mI1 - mI0, EMA_ALPHA) / (0.5 * (vI0 + vI1))
        wmQ = ema_1d(mQ1 - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQ1))
        if relax.sum() >= 50:
            mIR, mQR, vIR, vQR = Iq[relax].mean(0), Qq[relax].mean(0), Iq[relax].var(0), Qq[relax].var(0)
            wrI = ema_1d(mIR - mI0, EMA_ALPHA) / (0.5 * (vI0 + vIR))
            wrQ = ema_1d(mQR - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQR))
        else:
            wrI = wrQ = np.zeros(T)
        premix(q, wmI, wmQ, wrI, wrQ, W, IF, t_sec)
    return W, relax_counts


def quantize_tables(Wf, bits):
    """One power-of-two scale per (qubit, feature), shared by its A and B words."""
    lim = 2 ** (bits - 1) - 1
    Wq = np.zeros_like(Wf)
    shifts = np.zeros(Wf.shape[:2], dtype=int)
    for q in range(Wf.shape[0]):
        for k in range(4):
            m = np.abs(Wf[q, k]).max()
            s = int(np.floor(np.log2(lim / m))) if m > 0 else 0
            Wq[q, k] = np.clip(np.rint(Wf[q, k] * 2.0 ** s), -lim - 1, lim)
            shifts[q, k] = s
    return Wq, shifts


def accumulate(I_raw, Q_raw, W, chunk=5000):
    """acc[n,q,k] = sum_t I*A + Q*B.  Exact for integer inputs/tables (all partial sums < 2^53)."""
    N, T = I_raw.shape
    n_q = W.shape[0]
    out = np.empty((N, n_q, 4))
    WA = W[:, :, 0, :].reshape(n_q * 4, T).T
    WB = W[:, :, 1, :].reshape(n_q * 4, T).T
    for s_ in range(0, N, chunk):
        e = min(s_ + chunk, N)
        out[s_:e] = (I_raw[s_:e].astype(np.float64) @ WA + Q_raw[s_:e].astype(np.float64) @ WB).reshape(e - s_, n_q, 4)
    return out


def acc_int(I_raw, Q_raw, Wq):
    return np.rint(accumulate(I_raw, Q_raw, Wq)).astype(np.int64)


# ====================================================================== classifier structure
def pair_lists(n_q):
    """'poly' variant products per qubit, as flat indices into V_flat (feature f of qubit q = q*4+f)."""
    out = []
    for q in range(n_q):
        pr = [(q * 4 + i, q * 4 + j) for i in range(4) for j in range(i, 4)]
        for j in range(n_q):
            if j != q:
                pr += [(q * 4 + 0, j * 4 + 0), (q * 4 + 0, j * 4 + 1), (q * 4 + 1, j * 4 + 0), (q * 4 + 1, j * 4 + 1)]
        out.append(pr)
    return out


def build_O(Vf, pairs, mul, one):
    parts = [np.full((len(Vf), 1), one, dtype=Vf.dtype), Vf]
    if pairs:
        parts.append(np.stack([mul(Vf[:, i], Vf[:, j]) for i, j in pairs], 1))
    return np.concatenate(parts, 1)


def ridge(O, y):
    t = 2.0 * y - 1.0
    A = O.T @ O + LAMBDA_REL * len(O) * np.eye(O.shape[1])
    return np.linalg.solve(A, O.T @ t)


def best_threshold(s, y):
    order = np.argsort(s)
    ss, ys = s[order], y[order]
    n0 = (ys == 0).sum()
    c1 = np.r_[0, np.cumsum(ys == 1)]
    c0 = np.r_[0, np.cumsum(ys == 0)]
    k = int(np.argmin(c1 + (n0 - c0)))
    if k == 0:
        return ss[0] - 1.0
    if k == len(ss):
        return ss[-1]
    return 0.5 * (ss[k - 1] + ss[k])


def fid_pred(pred, y):
    return 0.5 * (np.mean(~pred[y == 0]) + np.mean(pred[y == 1]))


# ====================================================================== float reference (same structure, no rounding)
def run_float(Wf, d, variant):
    Ic, Qc, Yc = d["clf"]
    It, Qt, Yt = d["te"]
    n_q = Yc.shape[1]
    Ac, At = accumulate(Ic, Qc, Wf), accumulate(It, Qt, Wf)
    mu = Ac.mean(0)
    A = np.percentile(np.abs(Ac - mu), CLIP, axis=0)
    n = np.ceil(np.log2(np.maximum(A, 1e-12)))
    f = lambda a: (np.clip((a + 2.0 ** n - mu) / 2.0 ** (n + 1), 0, 1) - 0.5).reshape(len(a), -1)
    Vc, Vt = f(Ac), f(At)
    pairs = pair_lists(n_q) if variant == "poly" else [[]] * n_q
    corr = np.zeros(Yt.shape, bool)
    for q in range(n_q):
        mul = lambda a, b: a * b
        Oc, Ot = build_O(Vc, pairs[q], mul, 1.0), build_O(Vt, pairs[q], mul, 1.0)
        w = ridge(Oc, Yc[:, q])
        pred = (Ot @ w) > best_threshold(Oc @ w, Yc[:, q])
        corr[:, q] = pred == (Yt[:, q] == 1)
    return corr


# ====================================================================== fixed-point model
def norm_consts(Ac):
    mu = np.rint(Ac.mean(0)).astype(np.int64)
    A = np.percentile(np.abs(Ac - mu), CLIP, axis=0)
    n = np.ceil(np.log2(np.maximum(A, 1.0))).astype(np.int64)
    return mu, n


def normalize_int(acc, mu, n):
    assert int(n.max()) + 1 + F_BITS < 62, "normalizer would overflow the int64 model"
    one = np.int64(1)
    tmp = np.clip(acc + (one << n) - mu, 0, one << (n + 1))
    u = (tmp << F_BITS) >> (n + 1)
    return u - (one << (F_BITS - 1))


MUL_INT = lambda a, b: (a * b) >> F_BITS


def fit_fixed(Vc_flat, Yc, variant, n_q):
    pairs = pair_lists(n_q) if variant == "poly" else [[]] * n_q
    lim = 2 ** (W_BITS - 1)
    models = []
    for q in range(n_q):
        Oc = build_O(Vc_flat, pairs[q], MUL_INT, np.int64(1 << F_BITS))
        wf = ridge(Oc / 2.0 ** F_BITS, Yc[:, q])
        rng_max = (lim - 1) / 2.0 ** W_FRAC
        wf = wf * 2.0 ** np.floor(np.log2(rng_max / np.abs(wf).max()))      # positive scaling never changes the decision
        wq = np.clip(np.rint(wf * 2.0 ** W_FRAC), -lim, lim - 1).astype(np.int64)
        thr = int(round(best_threshold((Oc @ wq).astype(np.float64), Yc[:, q])))
        models.append(dict(w=wq, thr=thr, pairs=pairs[q]))
    return models


def apply_fixed(V_flat, models):
    scores = np.empty((len(V_flat), len(models)), np.int64)
    for q, m in enumerate(models):
        O = build_O(V_flat, m["pairs"], MUL_INT, np.int64(1 << F_BITS))
        scores[:, q] = O @ m["w"]
    thr = np.array([m["thr"] for m in models], dtype=np.int64)
    return scores, scores > thr[None, :]


# ====================================================================== .mem writers
def hexw(x, bits):
    return format(int(x) & ((1 << bits) - 1), "0{}x".format((bits + 3) // 4))


def write_mem(path, words, comment):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write("// " + comment + "\n")
        f.write("\n".join(words) + "\n")


def freeze_mode(mode):
    print(f"\n==================== MODE: {mode} ====================")
    d = get_data(mode)
    Ic, Qc, Yc = d["clf"]
    T_mode = Ic.shape[1]
    n_q = Yc.shape[1]
    out = os.path.join(OUT_DIR, mode)
    os.makedirs(out, exist_ok=True)
    print(d["desc"])
    print(f"CAL={len(d['cal'][2])}  CLF={len(Yc)}  TEST={len(d['te'][2])}  qubits={n_q}  T={T_mode}")

    # --- calibrate + quantize the front end
    Wf, relax_counts = calibrate(*d["cal"], d["IF"])
    print(f"relaxation traces labeled per qubit: {relax_counts}")
    Wq, shifts = quantize_tables(Wf, DLO_BITS)
    S_k = np.abs(Wq[:, :, 0, :]).sum(2) + np.abs(Wq[:, :, 1, :]).sum(2)                 # (q,k) sum of |words|
    acc_bits = int(np.ceil(np.log2(2 ** (INPUT_BITS - 1) * S_k.max() + 1))) + 1       # worst-case, not empirical
    assert acc_bits <= MEM_WORD, "accumulator wider than MEM_WORD"
    zero_cols = int((np.abs(Wq).max(axis=3) == 0).sum())

    Acc_c, Acc_t = acc_int(Ic, Qc, Wq), acc_int(*d["te"][:2], Wq)
    emp_bits = int(np.ceil(np.log2(max(np.abs(Acc_c).max(), np.abs(Acc_t).max()) + 1))) + 1
    mu, n = norm_consts(Acc_c)
    Vc = normalize_int(Acc_c, mu, n).reshape(len(Acc_c), -1)
    Vt = normalize_int(Acc_t, mu, n).reshape(len(Acc_t), -1)
    print(f"accumulator: worst-case {acc_bits} bits (used for the hardware), empirical {emp_bits} bits; "
          f"n = {n.reshape(-1).tolist()}")

    # --- train + evaluate each classifier variant
    Yt = d["te"][2]
    results, models_by_variant = {}, {}
    for variant in VARIANTS:
        models = fit_fixed(Vc, Yc, variant, n_q)
        models_by_variant[variant] = models
        _, pred = apply_fixed(Vt, models)
        corr_fix = pred == (Yt == 1)
        corr_flt = run_float(Wf, d, variant)
        fixed_F = np.mean([fid_pred(pred[:, q], Yt[:, q]) for q in range(n_q)])
        dd = corr_fix.mean(1).astype(float) - corr_flt.mean(1).astype(float)
        loss, se = 100 * dd.mean(), 100 * dd.std() / np.sqrt(len(dd))
        ok = (loss + 2 * se) >= -ALLOWED_LOSS_PTS      # flag only if clearly beyond the budget, not just noisy
        n_w = len(models[0]["w"])
        n_p = len(models[0]["pairs"])
        results[variant] = dict(fixed_fidelity=float(fixed_F), fixed_minus_float_pts=float(loss), se_pts=float(se),
                                n_weights=n_w, n_products=n_p, within_budget=bool(ok))
        print(f"[{variant:<4s}] fixed F={fixed_F:.4f}  fixed-minus-float = {loss:+.3f} +/- {se:.3f} pts  "
              f"({n_w} weights, {n_p} products per qubit)  {'OK' if ok else '*** LOSS TOO LARGE: widen bits ***'}")

    # --- export common front-end files
    mask_note = f"{DLO_BITS}-bit signed words, 8 per line packed A0|B0|A1|B1|A2|B2|A3|B3 (A0 in the least-significant bits)"
    for q in range(n_q):
        lines = []
        for t in range(T_mode):
            v = 0
            for k in range(4):
                for ab in range(2):
                    v |= (int(Wq[q, k, ab, t]) & ((1 << DLO_BITS) - 1)) << ((2 * k + ab) * DLO_BITS)
            lines.append(format(v, "0{}x".format(2 * DLO_BITS)))
        write_mem(os.path.join(out, f"dlo_q{q}.mem"), lines, f"qubit {q} weight table, depth {T_mode}, {mask_note}")
        write_mem(os.path.join(out, f"norm_mu_q{q}.mem"), [hexw(mu[q, k], MEM_WORD) for k in range(4)],
                  f"qubit {q} normalizer centers mu, 4 features, {MEM_WORD}-bit signed")
        write_mem(os.path.join(out, f"norm_n_q{q}.mem"), [hexw(n[q, k], 8) for k in range(4)],
                  f"qubit {q} normalizer exponents n, 4 features, 8-bit unsigned")

    # --- export classifier files
    var_cfg = {}
    for variant in VARIANTS:
        models = models_by_variant[variant]
        score_bits = 0
        for q, m in enumerate(models):
            n_v = n_q * 4
            max_o = np.r_[2 ** F_BITS, np.full(n_v, 2 ** (F_BITS - 1)), np.full(len(m["pairs"]), 2 ** (F_BITS - 2))]
            bound = float((max_o * np.abs(m["w"])).sum())
            score_bits = max(score_bits, int(np.ceil(np.log2(bound + 1))) + 1)
            write_mem(os.path.join(out, variant, f"cls_w_q{q}.mem"), [hexw(w, W_BITS) for w in m["w"]],
                      f"qubit {q} classifier weights ({variant}), order [bias, {n_v} features, {len(m['pairs'])} products], "
                      f"{W_BITS}-bit signed, {W_FRAC} fractional bits")
            write_mem(os.path.join(out, variant, f"cls_thr_q{q}.mem"), [hexw(m["thr"], MEM_WORD)],
                      f"qubit {q} decision threshold ({variant}), {MEM_WORD}-bit signed, scale 2^-{F_BITS + W_FRAC}")
        assert score_bits <= MEM_WORD
        var_cfg[variant] = dict(results[variant], score_bits=score_bits,
                                pairs=[[list(p) for p in m["pairs"]] for m in models])

    # --- golden vectors (+4 stress shots to exercise saturation and the accumulator extremes)
    It, Qt = d["te"][:2]
    mx = 2 ** (INPUT_BITS - 1) - 1
    alt = np.where(np.arange(T_mode) % 2 == 0, mx, -mx).astype(np.int16)
    stI = np.stack([np.full(T_mode, mx), np.full(T_mode, -mx), np.zeros(T_mode), alt]).astype(np.int16)
    stQ = np.stack([-stI[0], stI[0], np.zeros(T_mode), -alt]).astype(np.int16)
    G = min(N_GOLD_NPZ, len(It))
    Ig, Qg = np.concatenate([It[:G], stI]), np.concatenate([Qt[:G], stQ])
    Yg = np.concatenate([Yt[:G], np.zeros((4, n_q), int)])
    Ag = acc_int(Ig, Qg, Wq)
    assert np.abs(Ag).max() < 2 ** (acc_bits - 1), "stress shot exceeded the worst-case accumulator width"
    Vg = normalize_int(Ag, mu, n).reshape(len(Ag), -1)
    gold = dict(I=Ig, Q=Qg, label=Yg, acc=Ag, v=Vg.reshape(len(Vg), n_q, 4))
    for variant in VARIANTS:
        sc, pr = apply_fixed(Vg, models_by_variant[variant])
        gold[f"score_{variant}"], gold[f"pred_{variant}"] = sc, pr.astype(np.int8)
    os.makedirs(os.path.join(out, "golden"), exist_ok=True)
    np.savez(os.path.join(out, "golden", "golden_vectors.npz"), **gold)

    sel = list(range(min(N_GOLD_MEM, G))) + list(range(G, G + 4))
    gd = os.path.join(out, "golden")
    write_mem(os.path.join(gd, "tb_input.mem"),
              [format(((int(Qg[i, t]) & 0xFFFF) << 16) | (int(Ig[i, t]) & 0xFFFF), "08x") for i in sel for t in range(T_mode)],
              f"testbench input: shot-major, {T_mode} words per shot, word = {{Q[15:0], I[15:0]}} int16 each")
    write_mem(os.path.join(gd, "tb_label.mem"), [hexw(Yg[i, q], 4) for i in sel for q in range(n_q)],
              "true label per shot per qubit (last 4 shots are stress vectors, label meaningless)")
    write_mem(os.path.join(gd, "tb_acc.mem"), [hexw(Ag[i, q, k], MEM_WORD) for i in sel for q in range(n_q) for k in range(4)],
              f"expected accumulators: shot, qubit, feature order, {MEM_WORD}-bit signed")
    write_mem(os.path.join(gd, "tb_v.mem"), [hexw(gold["v"][i, q, k], 16) for i in sel for q in range(n_q) for k in range(4)],
              "expected normalized features v: shot, qubit, feature order, 16-bit signed (1.0 = 2^F)")
    for variant in VARIANTS:
        write_mem(os.path.join(gd, f"tb_score_{variant}.mem"),
                  [hexw(gold[f"score_{variant}"][i, q], MEM_WORD) for i in sel for q in range(n_q)],
                  f"expected scores ({variant}): shot, qubit order, {MEM_WORD}-bit signed")
        write_mem(os.path.join(gd, f"tb_pred_{variant}.mem"),
                  [hexw(gold[f"pred_{variant}"][i, q], 1) for i in sel for q in range(n_q)],
                  f"expected decision ({variant}): shot, qubit order, 1 bit")

    # --- config.json + design_spec.md
    cfg = dict(
        mode=mode, description=d["desc"], n_qubits=n_q, if_freqs_hz=[float(x) for x in d["IF"]],
        assumed_fs_hz=FS if mode in ("bus5", "spin_single", "spin_bus5") else None, T_bins=T_mode, T=T_mode,
        widths=dict(input_bits=INPUT_BITS, dlo_bits=DLO_BITS, acc_bits_worst_case=acc_bits, acc_bits_empirical=emp_bits,
                    F_bits=F_BITS, feature_bits=F_BITS + 1, w_bits=W_BITS, w_frac=W_FRAC, mem_word=MEM_WORD),
        filters=dict(ema_alpha=EMA_ALPHA, relax_traces=relax_counts, normalizer_clip_pct=CLIP),
        dlo_packing="word index = 2*feature + (0 for A multiplies I, 1 for B multiplies Q); A0 in the least-significant bits",
        feature_order="per qubit: [MF_I, MF_Q, RMF_I, RMF_Q]; flat index = qubit*4 + feature",
        classifier_input_order="[bias(=2^F), all qubits' features (flat order), products (pairs list)]",
        variants=var_cfg, golden=dict(shots_in_mem=len(sel), stress_shots=4, shots_in_npz=G + 4),
        all_zero_table_columns=zero_cols,
        not_verified=["latency and clock rate (no RTL yet)", "LUT/DSP/BRAM use (no synthesis yet)",
                      "real ADC sample period (QICK rate not confirmed; bins are not yet nanoseconds)",
                      "real multi-qubit crosstalk (bus5 is synthetic)", "separate contribution of WMF vs WRMF on the bus"],
    )
    with open(os.path.join(out, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    spec = [f"# Design freeze v1 - mode `{mode}`", "", d["desc"], "",
            "## Frozen parameters", f"- qubits: {n_q}, accumulation length T = {T_mode} bins, EMA alpha = {EMA_ALPHA}",
            f"- input {INPUT_BITS}-bit signed; weight-table words {DLO_BITS}-bit signed; accumulator {acc_bits}-bit signed (worst case, "
            f"empirical {emp_bits})",
            f"- normalized features: 1.0 = 2^{F_BITS}, range [-2^{F_BITS - 1}, +2^{F_BITS - 1}] ({F_BITS + 1}-bit signed)",
            f"- classifier weights {W_BITS}-bit signed ({W_FRAC} fractional bits); scores/threshold stored as {MEM_WORD}-bit signed", "",
            "## Integer algorithm (bit-exact reference: freeze_design.py / verify_mem_files.py)",
            "```", "acc[q][k] = sum_t ( I[t]*A[q][k][t] + Q[t]*B[q][k][t] )   k = MF_I, MF_Q, RMF_I, RMF_Q",
            "tmp = clip(acc + 2^n - mu, 0, 2^(n+1));  u = (tmp << F) >> (n+1);  v = u - 2^(F-1)",
            "product = (v_i * v_j) >> F      (arithmetic shift)", "score = sum_i w_i * o_i ;  o = [2^F, v_flat..., products...]",
            "decision = score > thr", "```", "",
            "## Memory per qubit", f"- `dlo_qN.mem`: depth {T_mode}, width {8 * DLO_BITS} bits (8 packed words)",
            "- `norm_mu_qN.mem`, `norm_n_qN.mem`: 4 entries each", "- `<variant>/cls_w_qN.mem`, `cls_thr_qN.mem`", "",
            "## Hardware work per sample per qubit",
            f"- 8 multiply-accumulates (4 features x 2 terms); {zero_cols} table columns are identically zero in this mode "
            f"(baseband single-qubit data drops to 4 effective MACs)", "",
            "## Measured on the TEST split (float vs fixed)"]
    for variant in VARIANTS:
        r = results[variant]
        spec.append(f"- `{variant}`: fixed F = {r['fixed_fidelity']:.4f}, fixed-minus-float = {r['fixed_minus_float_pts']:+.3f} "
                    f"+/- {r['se_pts']:.3f} pts, {r['n_weights']} weights + {r['n_products']} products per qubit")
    spec += ["", "## NOT verified yet"] + [f"- {x}" for x in cfg["not_verified"]]
    with open(os.path.join(out, "design_spec.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(spec) + "\n")
    print(f"wrote {out}/  (config.json, design_spec.md, dlo/norm/cls .mem files, golden/)")
    return results


all_results = {m: freeze_mode(m) for m in MODES}
print("\n==================== FREEZE SUMMARY ====================")
for m, r in all_results.items():
    for v, x in r.items():
        print(f"{m:<11s} {v:<5s} F={x['fixed_fidelity']:.4f}  loss vs float {x['fixed_minus_float_pts']:+.3f} pts  "
              f"{'OK' if x['within_budget'] else 'TOO LARGE'}")
print(f"\nNext: python pack_unified_profiles.py")

loading source-shot pools ...
peak |sample| = 4883  (14 signed bits needed; container = 16)

==================== MODE: single ====================
REAL QICK single-qubit data (Zenodo 14427490), baseband (already downconverted)
CAL=100000  CLF=100000  TEST=100000  qubits=1  T=500
relaxation traces labeled per qubit: [2576]
accumulator: worst-case 34 bits (used for the hardware), empirical 28 bits; n = [26, 26, 25, 25]
[lin ] fixed F=0.9610  fixed-minus-float = -0.001 +/- 0.001 pts  (5 weights, 0 products per qubit)  OK
[poly] fixed F=0.9613  fixed-minus-float = +0.004 +/- 0.004 pts  (15 weights, 10 products per qubit)  OK
wrote design_freeze\single/  (config.json, design_spec.md, dlo/norm/cls .mem files, golden/)

==================== MODE: bus5 ====================
SYNTHETIC 5-qubit multiplexed bus built from real single-qubit shots, crosstalk strength 1.0
CAL=30000  CLF=50000  TEST=100000  qubits=5  T=500
relaxation traces labeled per qubit: [1256, 1502, 1554, 1491, 1388]
accumulator